# Production Patterns for Agent Protocols

Prototype agent protocol integrations work in a notebook. Production deployments need **versioned contracts**, **health probes**, **circuit breakers**, **rate limits**, **distributed tracing**, and a **stable service boundary** between your HTTP API and protocol clients (MCP, A2A).

This capstone assembles those patterns for **ShopCo Support Hub** — a protocol gateway that connects to internal MCP servers and external A2A partner agents, then exposes a single `run_support()` contract to callers.

**What you will build:**

- `ProtocolGateway` — startup registry of MCP + A2A connections with qualified tool routing
- Health and readiness probes per protocol endpoint
- Circuit breakers, retries, and per-tenant rate limits
- Correlation-ID tracing across MCP `tools/call` and A2A task hops
- Schema version negotiation and compatibility checks
- `ShopCoProtocolService` + simulated API (`/health`, `/ready`, `/support/run`)
- Canary routing for rolling out new MCP server versions

Everything runs offline in plain Python. Set `USE_LIVE_LLM = True` only when you want a real model to drive tool selection through the gateway.

In [ ]:
"""
ShopCo Protocol Gateway — production patterns capstone.
"""

import json
import re
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()


# --- Backing stores ---
ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "customer": "alice@shop.com", "status": "shipped", "total_usd": 1299.0, "partner_ref": "SF-77821"},
    "ORD-1002": {"order_id": "ORD-1002", "customer": "bob@shop.com", "status": "processing", "total_usd": 449.0, "partner_ref": None},
}
POLICIES: dict[str, str] = {"returns": "30-day returns on unused items.", "shipping": "Free shipping over $50."}
PARTNER_SHIPMENTS: dict[str, dict[str, Any]] = {"SF-77821": {"status": "delivered_late", "days_late": 7}}
REFUND_QUEUE: list[dict[str, Any]] = []

TRACE_STORE: list[dict[str, Any]] = []
METRICS: dict[str, list[float]] = {"mcp_latency_ms": [], "a2a_latency_ms": [], "errors": []}

print(f"ShopCo protocol lab: {len(ORDERS)} orders, tracing enabled")

---

## 1. Production Protocol Stack

```
┌──────────────────────────────────────────────────────────────┐
│  L7  API            GET /health  GET /ready  POST /support/run│
├──────────────────────────────────────────────────────────────┤
│  L6  Service        ShopCoProtocolService.run_support()      │
├──────────────────────────────────────────────────────────────┤
│  L5  Gateway        ProtocolGateway — route, guard, trace    │
├──────────────────────────────────────────────────────────────┤
│  L4  Resilience     circuit breakers, retries, rate limits   │
├──────────────────────────────────────────────────────────────┤
│  L3  MCP clients    orders, policies (internal)              │
│  L3  A2A clients    shipfast partner agent (external)        │
├──────────────────────────────────────────────────────────────┤
│  L2  Protocol       JSON-RPC (MCP) + A2A task envelopes      │
├──────────────────────────────────────────────────────────────┤
│  L1  Observability  correlation IDs, spans, metrics          │
└──────────────────────────────────────────────────────────────┘
```

**Key rule:** compile protocol connections and tool manifests **once at startup**. Per-request work is routing, policy checks, and invocation — not rediscovery of the entire mesh.

---

## 2. Protocol Endpoint Registry

Each connection — MCP server or A2A partner — is registered with metadata production ops needs: version, transport, health probe, and capability manifest.

In [ ]:
class Transport(str, Enum):
    IN_PROCESS = "in_process"
    STDIO = "stdio"
    HTTP_SSE = "http_sse"


class EndpointKind(str, Enum):
    MCP = "mcp"
    A2A = "a2a"


@dataclass
class ProtocolEndpoint:
    alias: str
    kind: EndpointKind
    name: str
    version: str
    transport: Transport
    protocol_version: str
    tools: list[str] = field(default_factory=list)
    healthy: bool = True
    canary_weight: float = 0.0  # 0 = stable only; 0.1 = 10% to canary

    def readiness(self) -> dict[str, Any]:
        return {
            "alias": self.alias,
            "kind": self.kind.value,
            "version": self.version,
            "healthy": self.healthy,
            "tools": len(self.tools),
        }


ENDPOINTS: dict[str, ProtocolEndpoint] = {
    "orders": ProtocolEndpoint("orders", EndpointKind.MCP, "shopco-orders", "2.1.0", Transport.IN_PROCESS, "2024-11-05", ["get_order", "search_orders"]),
    "policies": ProtocolEndpoint("policies", EndpointKind.MCP, "shopco-policies", "1.0.0", Transport.IN_PROCESS, "2024-11-05", ["search_policies"]),
    "shipfast": ProtocolEndpoint("shipfast", EndpointKind.A2A, "shipfast-logistics", "1.4.0", Transport.HTTP_SSE, "2024-11-05", ["get_shipment_status"]),
    "orders_canary": ProtocolEndpoint("orders_canary", EndpointKind.MCP, "shopco-orders", "2.2.0-beta", Transport.IN_PROCESS, "2024-11-05", ["get_order", "search_orders", "get_order_v2"], canary_weight=0.0),
}

print("Registered endpoints:")
for ep in ENDPOINTS.values():
    print(f"  • {ep.alias} ({ep.kind.value}) v{ep.version} — {ep.tools}")

---

## 3. In-Process MCP and A2A Runtimes

The gateway talks to protocol handlers, not raw databases. Internal MCP servers wrap ShopCo data; the A2A handler wraps partner shipment data.

In [ ]:
class MCPServer:
    def __init__(self, name: str, version: str):
        self.name = name
        self.version = version
        self._tools: dict[str, Callable[..., Any]] = {}

    def tool(self, name: str):
        def decorator(fn: Callable):
            self._tools[name] = fn
            return fn
        return decorator

    def call(self, name: str, args: dict[str, Any]) -> Any:
        fn = self._tools.get(name)
        if not fn:
            raise ValueError(f"Unknown tool: {name}")
        return fn(**args)

    def initialize(self) -> dict[str, Any]:
        return {"serverInfo": {"name": self.name, "version": self.version}, "protocolVersion": "2024-11-05"}


orders_mcp = MCPServer("shopco-orders", "2.1.0")

@orders_mcp.tool("get_order")
def get_order(order_id: str) -> dict[str, Any]:
    o = ORDERS.get(order_id.strip().upper())
    if not o:
        raise ValueError(f"Order {order_id} not found")
    return o

@orders_mcp.tool("search_orders")
def search_orders(customer: str) -> list[dict[str, Any]]:
    return [o for o in ORDERS.values() if customer.lower() in o["customer"].lower()]


orders_canary_mcp = MCPServer("shopco-orders", "2.2.0-beta")

@orders_canary_mcp.tool("get_order")
def get_order_stable(order_id: str) -> dict[str, Any]:
    return get_order(order_id)

@orders_canary_mcp.tool("get_order_v2")
def get_order_v2(order_id: str, include_partner_ref: bool = True) -> dict[str, Any]:
    """V2 schema adds partner_ref and schema_version field."""
    data = get_order(order_id)
    data["schema_version"] = "2.2"
    if not include_partner_ref:
        data.pop("partner_ref", None)
    return data


policies_mcp = MCPServer("shopco-policies", "1.0.0")

@policies_mcp.tool("search_policies")
def search_policies(query: str) -> list[dict[str, str]]:
    q = query.lower()
    return [{"id": k, "text": v} for k, v in POLICIES.items() if q in k or q in v.lower()]


class A2APartnerAgent:
    def __init__(self, org: str, version: str):
        self.org = org
        self.version = version

    def handle_task(self, skill: str, payload: dict[str, Any]) -> dict[str, Any]:
        if skill == "get_shipment_status":
            ref = payload.get("shipment_id", "")
            rec = PARTNER_SHIPMENTS.get(ref)
            if not rec:
                raise ValueError(f"Shipment {ref} not found")
            return {"shipment_id": ref, **rec}
        raise ValueError(f"Unknown skill: {skill}")


shipfast_a2a = A2APartnerAgent("shipfast", "1.4.0")
MCP_BACKENDS = {"orders": orders_mcp, "orders_canary": orders_canary_mcp, "policies": policies_mcp}
A2A_BACKENDS = {"shipfast": shipfast_a2a}
print("Protocol backends online")

---

## 4. Observability — Correlation IDs and Spans

Every support request gets a `correlation_id`. Each MCP `tools/call` and A2A task creates a **span** with latency, endpoint alias, and outcome. This is how you debug cross-protocol flows in production.

In [ ]:
@dataclass
class Span:
    span_id: str
    correlation_id: str
    operation: str
    endpoint: str
    started_at: str
    duration_ms: float = 0.0
    status: str = "ok"
    detail: dict[str, Any] = field(default_factory=dict)


class Tracer:
    def start_span(self, correlation_id: str, operation: str, endpoint: str) -> Span:
        return Span(str(uuid.uuid4())[:8], correlation_id, operation, endpoint, utcnow())

    def end_span(self, span: Span, status: str = "ok", detail: dict[str, Any] | None = None) -> None:
        span.status = status
        span.detail = detail or {}
        started = datetime.fromisoformat(span.started_at)
        span.duration_ms = (datetime.now(timezone.utc) - started).total_seconds() * 1000
        TRACE_STORE.append({
            "span_id": span.span_id, "correlation_id": span.correlation_id,
            "operation": span.operation, "endpoint": span.endpoint,
            "duration_ms": round(span.duration_ms, 2), "status": span.status, "detail": span.detail,
        })


tracer = Tracer()
print("Tracer ready")

---

## 5. Resilience — Circuit Breaker, Retry, Rate Limit

| Pattern | Purpose |
|---------|--------|
| **Circuit breaker** | Stop calling a failing endpoint; fail fast until recovery window |
| **Retry with backoff** | Handle transient transport errors (not business errors) |
| **Rate limit** | Protect partner quotas and your own infra per tenant |

In [ ]:
class CircuitState(str, Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"


@dataclass
class CircuitBreaker:
    failure_threshold: int = 3
    recovery_seconds: float = 30.0
    failures: int = 0
    state: CircuitState = CircuitState.CLOSED
    opened_at: float | None = None

    def record_success(self) -> None:
        self.failures = 0
        self.state = CircuitState.CLOSED

    def record_failure(self) -> None:
        self.failures += 1
        if self.failures >= self.failure_threshold:
            self.state = CircuitState.OPEN
            self.opened_at = time.monotonic()

    def allow_call(self) -> bool:
        if self.state == CircuitState.CLOSED:
            return True
        if self.state == CircuitState.OPEN and self.opened_at:
            if time.monotonic() - self.opened_at >= self.recovery_seconds:
                self.state = CircuitState.HALF_OPEN
                return True
            return False
        return True  # half_open: allow probe


@dataclass
class TokenBucket:
    capacity: int
    tokens: float
    refill_per_sec: float
    last_refill: float = field(default_factory=time.monotonic)

    def consume(self, n: float = 1.0) -> bool:
        now = time.monotonic()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_per_sec)
        self.last_refill = now
        if self.tokens >= n:
            self.tokens -= n
            return True
        return False


BREAKERS: dict[str, CircuitBreaker] = {alias: CircuitBreaker() for alias in ENDPOINTS}
RATE_LIMITS: dict[str, TokenBucket] = {alias: TokenBucket(capacity=20, tokens=20, refill_per_sec=5) for alias in ENDPOINTS}


def retry_transient(fn: Callable[[], Any], max_attempts: int = 3, base_delay: float = 0.05) -> Any:
    last_err: Exception | None = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except ConnectionError as e:
            last_err = e
            time.sleep(base_delay * (2 ** attempt))
    raise last_err or ConnectionError("retry exhausted")

print("Resilience primitives ready")

---

## 6. ProtocolGateway — The Production Router

The gateway is compiled at startup. It maps qualified tool names (`orders__get_order`) to endpoints, enforces resilience policies, emits spans, and supports **canary routing** for MCP server upgrades.

In [ ]:
@dataclass
class ToolRoute:
    qualified_name: str
    endpoint_alias: str
    tool_name: str
    destructive: bool = False


class ProtocolGateway:
    def __init__(self):
        self._routes: dict[str, ToolRoute] = {}
        self._compiled = False

    def compile(self) -> None:
        """Run once at startup — discover tools from all endpoints."""
        manifest = [
            ("orders", "get_order", False), ("orders", "search_orders", False),
            ("policies", "search_policies", False),
            ("orders_canary", "get_order_v2", False),
        ]
        for alias, tool, destructive in manifest:
            qname = f"{alias.split('_')[0]}__{tool}" if "canary" not in alias else f"orders__{tool}"
            self._routes[qname] = ToolRoute(qname, alias, tool, destructive)
        self._a2a_skills = {"shipfast__get_shipment_status": ("shipfast", "get_shipment_status")}
        self._compiled = True

    def _pick_orders_endpoint(self) -> str:
        """Canary: route small % of traffic to orders_canary."""
        ep = ENDPOINTS["orders_canary"]
        if ep.canary_weight > 0 and ep.healthy:
            if uuid.uuid4().int % 100 < int(ep.canary_weight * 100):
                return "orders_canary"
        return "orders" if ENDPOINTS["orders"].healthy else "orders_canary"

    def invoke_mcp(self, correlation_id: str, qualified_name: str, args: dict[str, Any]) -> Any:
        if not self._compiled:
            raise RuntimeError("Gateway not compiled — call compile() at startup")
        route = self._routes.get(qualified_name)
        if not route:
            raise ValueError(f"Unknown tool: {qualified_name}")
        alias = route.endpoint_alias
        if qualified_name.startswith("orders__get_order"):
            alias = self._pick_orders_endpoint()
            tool = "get_order_v2" if alias == "orders_canary" and route.tool_name == "get_order_v2" else route.tool_name.replace("_v2", "")
        else:
            tool = route.tool_name

        breaker = BREAKERS.get(alias) or BREAKERS.get(route.endpoint_alias)
        bucket = RATE_LIMITS.get(alias) or RATE_LIMITS[route.endpoint_alias]
        if breaker and not breaker.allow_call():
            METRICS["errors"].append(1.0)
            raise ConnectionError(f"Circuit open for {alias}")
        if not bucket.consume():
            raise ConnectionError(f"Rate limit exceeded for {alias}")

        span = tracer.start_span(correlation_id, "mcp.tools/call", alias)
        try:
            server = MCP_BACKENDS[alias]
            result = retry_transient(lambda: server.call(tool, args))
            breaker.record_success()
            METRICS["mcp_latency_ms"].append(span.duration_ms)
            tracer.end_span(span, "ok", {"tool": tool, "qualified": qualified_name})
            return result
        except Exception as e:
            breaker.record_failure()
            METRICS["errors"].append(1.0)
            tracer.end_span(span, "error", {"error": str(e)})
            raise

    def invoke_a2a(self, correlation_id: str, qualified_skill: str, payload: dict[str, Any]) -> Any:
        mapping = self._a2a_skills.get(qualified_skill)
        if not mapping:
            raise ValueError(f"Unknown A2A skill: {qualified_skill}")
        alias, skill = mapping
        breaker = BREAKERS[alias]
        bucket = RATE_LIMITS[alias]
        if not breaker.allow_call():
            raise ConnectionError(f"Circuit open for {alias}")
        if not bucket.consume():
            raise ConnectionError(f"Rate limit exceeded for {alias}")
        span = tracer.start_span(correlation_id, "a2a.task", alias)
        try:
            agent = A2A_BACKENDS[alias]
            result = agent.handle_task(skill, payload)
            breaker.record_success()
            tracer.end_span(span, "ok", {"skill": skill})
            return result
        except Exception as e:
            breaker.record_failure()
            METRICS["errors"].append(1.0)
            tracer.end_span(span, "error", {"error": str(e)})
            raise

    def list_tools(self) -> list[str]:
        return list(self._routes.keys())

    def health(self) -> dict[str, Any]:
        return {"status": "ok", "gateway_compiled": self._compiled}

    def ready(self) -> dict[str, Any]:
        checks = [ep.readiness() for ep in ENDPOINTS.values() if ep.canary_weight == 0 or ep.alias == "orders"]
        all_healthy = all(c["healthy"] for c in checks if c["alias"] != "orders_canary")
        return {"ready": all_healthy and self._compiled, "endpoints": checks}


gateway = ProtocolGateway()
gateway.compile()
print(f"Gateway compiled: {gateway.list_tools()}")

---

## 7. Schema Version Negotiation

MCP servers evolve. Production hosts check `protocolVersion` at `initialize` and validate that required tool schemas are present before routing traffic. Canary endpoints let you test `get_order_v2` without breaking stable callers.

In [ ]:
REQUIRED_PROTOCOL = "2024-11-05"
REQUIRED_TOOLS_STABLE = {"get_order", "search_policies"}


def negotiate_mcp(server: MCPServer, required_tools: set[str]) -> tuple[bool, list[str]]:
    init = server.initialize()
    if init.get("protocolVersion") != REQUIRED_PROTOCOL:
        return False, [f"protocol mismatch: {init.get('protocolVersion')}"]
    available = set(server._tools.keys())
    missing = required_tools - available
    return len(missing) == 0, list(missing)


for alias, server in MCP_BACKENDS.items():
    required = {"search_policies"} if alias == "policies" else {"get_order"}
    ok, issues = negotiate_mcp(server, required)
    print(f"{alias}: negotiate={'OK' if ok else 'FAIL'}" + (f" missing={issues}" if issues else ""))

---

## 8. Approval Gate for Destructive Protocol Actions

Protocol gateways sit in front of MCP and A2A — the right place to enforce human-in-the-loop for mutating tools like `request_refund` before the call crosses any transport.

In [ ]:
@dataclass
class ApprovalPolicy:
    auto_approve_reads: bool = True
    approved_destructive: set[str] = field(default_factory=set)


DESTRUCTIVE_TOOLS = {"orders__request_refund"}


class GuardedGateway:
    def __init__(self, inner: ProtocolGateway, policy: ApprovalPolicy):
        self.inner = inner
        self.policy = policy

    def invoke_mcp(self, correlation_id: str, qualified_name: str, args: dict[str, Any], session_id: str = "") -> Any:
        if qualified_name in DESTRUCTIVE_TOOLS:
            key = f"{session_id}:{qualified_name}"
            if key not in self.policy.approved_destructive:
                raise PermissionError(f"Destructive tool {qualified_name} requires approval")
        return self.inner.invoke_mcp(correlation_id, qualified_name, args)


guarded = GuardedGateway(gateway, ApprovalPolicy())
cid = "CORR-APPROVAL-DEMO"
try:
    guarded.invoke_mcp(cid, "orders__request_refund", {"order_id": "ORD-1001"})
except PermissionError as e:
    print(f"Blocked: {e}")
print("Gateway enforces HITL at protocol boundary — not inside the LLM prompt.")

---

## 9. ShopCoProtocolService — Stable Service Contract

HTTP handlers and workers call one method: `run_support()`. They never touch MCP clients or A2A envelopes directly.

In [ ]:
@dataclass
class SupportResult:
    correlation_id: str
    reply: str
    tools_used: list[str]
    partner_data: dict[str, Any] = field(default_factory=dict)
    errors: list[str] = field(default_factory=list)


class ShopCoProtocolService:
    def __init__(self, gw: ProtocolGateway):
        self.gateway = gw

    def run_support(self, user_message: str, tenant_id: str = "default") -> SupportResult:
        correlation_id = f"SUP-{uuid.uuid4().hex[:8].upper()}"
        tools_used: list[str] = []
        errors: list[str] = []
        partner_data: dict[str, Any] = {}

        oid_match = re.search(r"ORD-\d+", user_message.upper())
        if not oid_match:
            return SupportResult(correlation_id, "Please provide an order ID.", tools_used)
        order_id = oid_match.group()

        try:
            order = self.gateway.invoke_mcp(correlation_id, "orders__get_order", {"order_id": order_id})
            tools_used.append("orders__get_order")
        except Exception as e:
            return SupportResult(correlation_id, f"Could not load order: {e}", tools_used, errors=[str(e)])

        if "policy" in user_message.lower() or "return" in user_message.lower():
            try:
                policies = self.gateway.invoke_mcp(correlation_id, "policies__search_policies", {"query": "return"})
                tools_used.append("policies__search_policies")
                policy_text = policies[0]["text"] if policies else "No policy found."
            except Exception as e:
                errors.append(f"policies: {e}")
                policy_text = "Policy service unavailable."
        else:
            policy_text = ""

        if order.get("partner_ref") and any(w in user_message.lower() for w in ("late", "ship", "delivery")):
            try:
                shipment = self.gateway.invoke_a2a(
                    correlation_id, "shipfast__get_shipment_status", {"shipment_id": order["partner_ref"]}
                )
                tools_used.append("shipfast__get_shipment_status")
                partner_data["shipment"] = shipment
            except Exception as e:
                errors.append(f"shipfast: {e}")

        reply_parts = [f"Order {order_id}: status={order['status']}, total=${order['total_usd']}"]
        if policy_text:
            reply_parts.append(f"Policy: {policy_text}")
        if "shipment" in partner_data:
            s = partner_data["shipment"]
            reply_parts.append(f"Shipment: {s['status']} ({s.get('days_late', 0)} days late)")
        if errors:
            reply_parts.append(f"Partial data — errors: {', '.join(errors)}")

        return SupportResult(correlation_id, "\n".join(reply_parts), tools_used, partner_data, errors)


service = ShopCoProtocolService(gateway)
demo = service.run_support("ORD-1001 arrived late, can I return it?")
print(f"Correlation: {demo.correlation_id}")
print(demo.reply)
print(f"\nTools used: {demo.tools_used}")

---

## 10. Simulated HTTP API

Production exposes health for liveness and readiness for traffic. Readiness checks that the gateway is compiled and critical endpoints are healthy.

In [ ]:
class SupportAPI:
    def __init__(self, svc: ShopCoProtocolService, gw: ProtocolGateway):
        self.service = svc
        self.gateway = gw

    def handle(self, method: str, path: str, body: dict[str, Any] | None = None) -> tuple[int, dict[str, Any]]:
        if method == "GET" and path == "/health":
            return 200, self.gateway.health()
        if method == "GET" and path == "/ready":
            r = self.gateway.ready()
            return (200 if r["ready"] else 503, r)
        if method == "POST" and path == "/support/run":
            msg = (body or {}).get("message", "")
            tenant = (body or {}).get("tenant_id", "default")
            result = self.service.run_support(msg, tenant)
            return 200, {
                "correlation_id": result.correlation_id,
                "reply": result.reply,
                "tools_used": result.tools_used,
                "errors": result.errors,
            }
        return 404, {"error": "not found"}


api = SupportAPI(service, gateway)
for req in [
    ("GET", "/health", None),
    ("GET", "/ready", None),
    ("POST", "/support/run", {"message": "Return policy for ORD-1001?"}),
]:
    status, payload = api.handle(*req)
    print(f"{req[0]} {req[1]} → {status}")
    print(f"  {json.dumps(payload, default=str)[:120]}..." if len(json.dumps(payload)) > 120 else f"  {payload}")
    print()

---

## 11. Canary Rollout for MCP Server Upgrades

Enable canary weight to shift a percentage of `orders__get_order` calls to the v2.2 server. Monitor error rate and latency before promoting.

In [ ]:
def simulate_canary_traffic(n: int = 50, weight: float = 0.2) -> dict[str, int]:
    ENDPOINTS["orders_canary"].canary_weight = weight
    counts = {"orders": 0, "orders_canary": 0}
    for _ in range(n):
        ep = gateway._pick_orders_endpoint()
        counts[ep] = counts.get(ep, 0) + 1
    return counts


distribution = simulate_canary_traffic(100, weight=0.25)
print(f"Canary 25% target — observed over 100 picks: {distribution}")
ENDPOINTS["orders_canary"].canary_weight = 0.0  # reset after demo

---

## 12. Failure Drill — Circuit Breaker Opens

Simulate partner failures until the circuit opens. The service returns partial results instead of hanging.

In [ ]:
class FailingA2AAgent(A2APartnerAgent):
    def handle_task(self, skill: str, payload: dict[str, Any]) -> dict[str, Any]:
        raise ConnectionError("ShipFast timeout")


A2A_BACKENDS["shipfast"] = FailingA2AAgent("shipfast", "1.4.0")
breaker = BREAKERS["shipfast"]
breaker.failures = 0
breaker.state = CircuitState.CLOSED

cid = "CORR-FAIL-DRILL"
for i in range(4):
    try:
        gateway.invoke_a2a(cid, "shipfast__get_shipment_status", {"shipment_id": "SF-77821"})
    except ConnectionError as e:
        print(f"  attempt {i+1}: {e}")

print(f"\nCircuit state: {breaker.state.value}")
result = service.run_support("ORD-1001 delivery was late")
print(f"Service degrades gracefully: errors={result.errors}")
print(result.reply)

# Restore healthy partner
A2A_BACKENDS["shipfast"] = shipfast_a2a
breaker.failures = 0
breaker.state = CircuitState.CLOSED

---

## 13. Trace Inspection

Filter spans by correlation ID to reconstruct a support request across MCP and A2A hops.

In [ ]:
def print_trace(correlation_id: str) -> None:
    spans = [s for s in TRACE_STORE if s["correlation_id"] == correlation_id]
    print(f"Trace {correlation_id} ({len(spans)} spans):")
    for s in spans:
        print(f"  [{s['operation']}] {s['endpoint']} {s['duration_ms']}ms {s['status']}")


print_trace(demo.correlation_id)

---

## 14. Metrics Dashboard (Simulated)

Aggregate in-memory metrics the way a production stack would export to Prometheus or Datadog.

In [ ]:
def metrics_summary() -> dict[str, Any]:
    def avg(vals: list[float]) -> float:
        return round(sum(vals) / len(vals), 2) if vals else 0.0
    return {
        "mcp_calls": len(METRICS["mcp_latency_ms"]),
        "mcp_avg_latency_ms": avg(METRICS["mcp_latency_ms"]),
        "a2a_calls": len(METRICS["a2a_latency_ms"]),
        "error_count": len(METRICS["errors"]),
        "trace_spans": len(TRACE_STORE),
        "circuits_open": [a for a, b in BREAKERS.items() if b.state == CircuitState.OPEN],
    }


print(pretty(metrics_summary()))

---

## 15. Configuration and Feature Flags

Protocol endpoints, canary weights, and approval policies belong in **configuration**, not agent prompts. Reload config without redeploying the LLM.

In [ ]:
@dataclass
class GatewayConfig:
    enable_shipfast_a2a: bool = True
    orders_canary_weight: float = 0.0
    max_tool_calls_per_request: int = 10
    require_correlation_id: bool = True


CONFIG = GatewayConfig(enable_shipfast_a2a=True, orders_canary_weight=0.1)


def apply_config(cfg: GatewayConfig) -> None:
    ENDPOINTS["orders_canary"].canary_weight = cfg.orders_canary_weight
    if not cfg.enable_shipfast_a2a:
        ENDPOINTS["shipfast"].healthy = False


apply_config(CONFIG)
print(f"Config applied: canary={CONFIG.orders_canary_weight}, shipfast={CONFIG.enable_shipfast_a2a}")
ENDPOINTS["orders_canary"].canary_weight = 0.0

---

## 16. Production Deployment Checklist

### Startup
- Compile gateway and negotiate protocol versions before accepting traffic
- `/ready` returns 503 until all critical endpoints pass health checks
- Warm MCP `initialize` sessions for stdio servers at boot

### Runtime
- Propagate `correlation_id` from API → gateway → MCP/A2A spans
- Circuit breakers per endpoint; alert when half-open probes fail
- Rate limits per tenant and per partner credential
- HITL gate at gateway for destructive tools

### Releases
- Canary new MCP server versions by tool routing weight
- Schema compatibility tests in CI before promoting manifests
- Rollback = set canary weight to 0 and mark unhealthy

### Security
- Scoped credentials for A2A; never embed partner secrets in prompts
- TLS for HTTP/SSE transports; stdio only for trusted local child processes
- Audit log every cross-org invocation with correlation ID

---

## 17. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| Discover tools per request | Latency, thundering herd | `compile()` at startup |
| No `/ready` probe | Traffic to broken MCP child process | Readiness checks all endpoints |
| Retry business errors | Duplicate refunds | Retry only transient transport errors |
| LLM holds partner credentials | Prompt leak risk | Gateway injects credentials |
| Single circuit for all partners | One outage blocks all | Per-endpoint circuit breakers |
| No correlation IDs | Un-debuggable cross-protocol flows | ID from API boundary onward |
| Big-bang MCP upgrades | Schema breaks hosts | Canary routing by weight |

---

## 18. Check Your Understanding

1. Why compile the protocol gateway at **startup** instead of per request?
2. What is the difference between `/health` (liveness) and `/ready` (readiness)?
3. When should a circuit breaker **open**, and what happens on the next call?
4. Why enforce HITL at the **gateway** rather than in the model prompt?
5. How does canary routing help roll out `get_order_v2` safely?
6. What belongs in a span for an MCP `tools/call`?
7. Why retry `ConnectionError` but not `ValueError` from a tool handler?

---

## 19. Optional — Live LLM Through the Gateway

Set `USE_LIVE_LLM = True` to let a model choose from gateway-discovered tools. The gateway still enforces rate limits, circuits, and tracing.

In [ ]:
if USE_LIVE_LLM:
    import os
    from openai import OpenAI

    tools = [
        {"type": "function", "function": {"name": n, "description": f"MCP tool {n}", "parameters": {"type": "object", "properties": {}}}}
        for n in gateway.list_tools()
    ]
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Status of ORD-1001 and return policy?"}],
        tools=tools,
    )
    print(resp.choices[0].message)
else:
    print("Offline mode — production gateway patterns demonstrated without live LLM.")

---

## 20. Summary

Production agent protocols need more than working JSON-RPC:

- **ProtocolGateway** compiles MCP + A2A routes once, qualifies tool names, and routes canary traffic
- **Resilience** — per-endpoint circuit breakers, token-bucket rate limits, transient retries
- **Observability** — correlation IDs and spans across every `tools/call` and A2A task
- **Service boundary** — `ShopCoProtocolService.run_support()` hides protocol complexity from HTTP/workers
- **Operations** — `/health`, `/ready`, schema negotiation, config-driven feature flags, and gradual MCP rollouts

Build the gateway layer first. Keep LLMs and HTTP handlers thin — they plan and present, while the protocol stack enforces policy, survives partner failures, and stays auditable.